In [ ]:
# Dataset: SynTTS-Commands (English, Free-ST subset)
# https://huggingface.co/datasets/lugan/Syntts-Commands-Media-Dataset
# Hanya 4 kelas playback control yang dipakai.

!pip install -q huggingface_hub

In [ ]:
import os
import zipfile
from huggingface_hub import hf_hub_download

REPO_ID = "lugan/Syntts-Commands-Media-Dataset"
SUBSET = "Free_ST_English"   # ~190 MB untuk 4 kelas. "VoxCeleb12_English" jauh lebih besar (~1.7 GB)

classes = [
    "Play",
    "Pause",
    "Next_track",
    "Previous_track"
]

DATASET_PATH = "/content/dataset/dataset"
os.makedirs(DATASET_PATH, exist_ok=True)

for label in classes:

    if os.path.isdir(os.path.join(DATASET_PATH, label)):
        print(f"{label}: sudah ada, skip")
        continue

    zip_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=f"{SUBSET}/{label}.zip"
    )

    # Setiap zip berisi folder <label>/*.wav
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATASET_PATH)

    print(f"{label}: {len(os.listdir(os.path.join(DATASET_PATH, label)))} file")

print("Dataset berhasil diunduh dan diekstrak!")

# ============================================================
# Kelas Unknown: rekaman noise (bukan dari Hugging Face)
# - Lokal (Jupyter dari folder Model_Train): dataset/noise_only
# - Colab: upload folder noise_only ke /content/dataset/noise_only
# ============================================================

NOISE_CANDIDATES = [
    "dataset/noise_only",
    "/content/dataset/noise_only"
]

NOISE_PATH = next(
    (p for p in NOISE_CANDIDATES if os.path.isdir(p)),
    None
)

if NOISE_PATH is None:
    raise FileNotFoundError(
        "Folder noise_only tidak ditemukan. Upload ke salah satu: "
        + ", ".join(NOISE_CANDIDATES)
    )

# Folder untuk setiap kelas
CLASS_DIRS = {
    label: os.path.join(DATASET_PATH, label)
    for label in classes
}

CLASS_DIRS["Unknown"] = NOISE_PATH

classes = classes + ["Unknown"]

print(f"Unknown: {len(os.listdir(NOISE_PATH))} file dari {NOISE_PATH}")

In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    level = root.replace("/content/dataset", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    if level >= 1:
        for file in files[:5]:
            print(f"{indent}    {file}")

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

records = []

for label in classes:

    folder = CLASS_DIRS[label]

    for filename in os.listdir(folder):

        if filename.lower().endswith((".wav", ".mp3", ".flac", ".ogg")):

            filepath = os.path.join(folder, filename)

            try:
                y, sr = librosa.load(filepath, sr=None, mono=False)

                if y.ndim == 1:
                    channels = 1
                    duration = len(y) / sr
                else:
                    channels = y.shape[0]
                    duration = y.shape[1] / sr

                records.append({
                    "file": filepath,
                    "label": label,
                    "sample_rate": sr,
                    "channels": channels,
                    "duration": duration
                })

            except Exception as e:
                print("ERROR:", filepath)
                print(e)

df = pd.DataFrame(records)

print(df.head())
print()
print("Jumlah data:", len(df))

In [ ]:
print(df["label"].value_counts())

In [ ]:
print(df["sample_rate"].value_counts())

In [ ]:
print(df["channels"].value_counts())

In [ ]:
print(df["duration"].describe())

In [ ]:
import os

for label in classes:
    folder = CLASS_DIRS[label]

    print(f"\n===== {label} =====")

    files = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".wav", ".mp3", ".flac", ".ogg"))
    ]

    for f in files[:20]:
        print(f)

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "metadata" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import re

def extract_speaker_id(filename):
    match = re.search(r'P(\d+)', filename)
    if match:
        return match.group(1)
    return None

In [ ]:
df["speaker_id"] = df["file"].apply(
    lambda x: extract_speaker_id(os.path.basename(x))
)

print(df[["file", "speaker_id", "label"]].head(20))

In [ ]:
print("Jumlah speaker:")
print(df["speaker_id"].nunique())

In [ ]:
speaker_command = df.groupby("speaker_id")["label"].nunique()

print(speaker_command.value_counts())

In [ ]:
print(
    df.groupby("speaker_id")["label"]
      .nunique()
      .sort_values(ascending=False)
      .head(20)
)

In [ ]:
example_speaker = df["speaker_id"].value_counts().index[0]

print(
    df[df["speaker_id"] == example_speaker]
    [["file", "speaker_id", "label"]]
    .to_string(index=False)
)

#Speaker-Independent Split

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

print("Jumlah data:", len(df))
print("\nDistribusi label:")
print(df["label"].value_counts())

# Unknown (noise) tidak punya speaker: setiap file noise menjadi
# grup sendiri, sehingga split command tetap speaker-independent
df["split_group"] = df["speaker_id"].fillna("noise_" + df["file"])

# ============================================================
# 1. SPLIT BERDASARKAN SPEAKER
#    70% Train, 30% Temp
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        df,
        y=df["label"],
        groups=df["split_group"]
    )
)

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

# ============================================================
# 2. Temp → Validation + Test
#    15% Validation, 15% Test
# ============================================================

gss_val_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss_val_test.split(
        temp_df,
        y=temp_df["label"],
        groups=temp_df["split_group"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# ============================================================
# 3. SHUFFLE
# ============================================================

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# ============================================================
# 4. HASIL SPLIT
# ============================================================

print("\n========== HASIL SPLIT ==========")

print(f"Train      : {len(train_df)} audio")
print(f"Validation : {len(val_df)} audio")
print(f"Test       : {len(test_df)} audio")

print("\n========== DISTRIBUSI LABEL ==========")

print("\nTRAIN:")
print(train_df["label"].value_counts())

print("\nVALIDATION:")
print(val_df["label"].value_counts())

print("\nTEST:")
print(test_df["label"].value_counts())

# ============================================================
# 5. CEK SPEAKER
# ============================================================

train_speakers = set(train_df["speaker_id"].dropna())
val_speakers = set(val_df["speaker_id"].dropna())
test_speakers = set(test_df["speaker_id"].dropna())

print("\n========== SPEAKER ==========")

print("Train speaker      :", len(train_speakers))
print("Validation speaker :", len(val_speakers))
print("Test speaker       :", len(test_speakers))

print("\n========== SPEAKER OVERLAP ==========")

print("Train ∩ Validation :", len(train_speakers & val_speakers))
print("Train ∩ Test       :", len(train_speakers & test_speakers))
print("Validation ∩ Test  :", len(val_speakers & test_speakers))

In [ ]:
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

print("Split berhasil disimpan.")

#audio preprocessing

In [ ]:
!pip install librosa soundfile

In [ ]:
TARGET_SR = 16000
DURATION = 1.0
TARGET_LENGTH = int(TARGET_SR * DURATION)

N_MFCC = 40
N_FFT = 512
HOP_LENGTH = 160
WIN_LENGTH = 400

print("Target sample rate :", TARGET_SR)
print("Target samples     :", TARGET_LENGTH)
print("MFCC               :", N_MFCC)
print("n_fft              :", N_FFT)
print("hop_length         :", HOP_LENGTH)
print("win_length         :", WIN_LENGTH)

In [ ]:
import librosa
import numpy as np

def preprocess_audio(file_path, label=None, random_crop=False):
    """
    Load dan preprocess audio:
    - Mono
    - Resample ke 16 kHz
    - Normalize amplitude
    - Audio menjadi 1 detik

    Untuk Unknown:
    - Jika audio > 1 detik, dapat mengambil random crop 1 detik.
    """

    # Load audio
    audio, sr = librosa.load(
        file_path,
        sr=TARGET_SR,
        mono=True
    )

    # Normalize amplitude
    max_val = np.max(np.abs(audio))

    if max_val > 0:
        audio = audio / max_val

    # ========================================================
    # Audio lebih pendek dari 1 detik
    # ========================================================

    if len(audio) < TARGET_LENGTH:

        pad_length = TARGET_LENGTH - len(audio)

        audio = np.pad(
            audio,
            (0, pad_length),
            mode="constant"
        )

    # ========================================================
    # Audio lebih panjang dari 1 detik
    # ========================================================

    elif len(audio) > TARGET_LENGTH:

        if label == "Unknown" and random_crop:

            # Random crop 1 detik
            max_start = len(audio) - TARGET_LENGTH

            start = np.random.randint(
                0,
                max_start + 1
            )

            audio = audio[
                start:start + TARGET_LENGTH
            ]

        else:

            # Untuk command:
            # tetap mengambil bagian awal
            audio = audio[:TARGET_LENGTH]

    return audio

In [ ]:
# Command
command_sample = train_df[
    train_df["label"] == "Play"
].iloc[0]

audio_command = preprocess_audio(
    command_sample["file"],
    label=command_sample["label"]
)

print("COMMAND")
print("File        :", command_sample["file"])
print("Shape       :", audio_command.shape)
print("Sample rate :", TARGET_SR)
print("Duration    :", len(audio_command) / TARGET_SR)
print("Min         :", audio_command.min())
print("Max         :", audio_command.max())

In [ ]:
import matplotlib.pyplot as plt
import librosa.display

plt.figure(figsize=(12, 4))

librosa.display.waveshow(
    audio_command,
    sr=TARGET_SR
)

plt.title("Waveform Command Setelah Preprocessing")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.show()

#MFCC

In [ ]:
def extract_mfcc(file_path, label=None, random_crop=False):
    """
    Preprocessing audio + ekstraksi MFCC.

    Output:
        shape = (N_MFCC, time_frames)
    """

    audio = preprocess_audio(
        file_path,
        label=label,
        random_crop=random_crop
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=TARGET_SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH
    )

    return mfcc

In [ ]:
# Ambil satu file COMMAND
command_row = train_df[train_df["label"] == "Play"].iloc[0]


# =========================
# COMMAND
# =========================
mfcc_command = extract_mfcc(
    command_row["file"],
    label=command_row["label"],
    random_crop=False
)

print("COMMAND")
print("File:", command_row["file"])
print("MFCC shape:", mfcc_command.shape)
print("MFCC min:", mfcc_command.min())
print("MFCC max:", mfcc_command.max())

In [ ]:
plt.figure(figsize=(10, 5))

librosa.display.specshow(
    mfcc_command,
    sr=TARGET_SR,
    hop_length=HOP_LENGTH,
    x_axis="time"
)

plt.colorbar(format="%+2.0f dB")
plt.title("MFCC - Play")
plt.xlabel("Time")
plt.ylabel("MFCC Coefficient")

plt.show()

In [ ]:
for label in classes:

    sample = train_df[
        train_df["label"] == label
    ].iloc[0]["file"]

    mfcc = extract_mfcc(sample)

    print(f"{label:15s} -> {mfcc.shape}")

In [ ]:
print("NaN :", np.isnan(mfcc).any())
print("Inf :", np.isinf(mfcc).any())

#Classical Speech Recognition Models

Empat model klasik dilatih dan dibandingkan:

| Model | Paradigma | Fitur |
|---|---|---|
| **GMM-HMM** | Generatif, sekuensial (satu HMM per kata) | MFCC+Δ+ΔΔ per frame |
| **DTW + kNN** | Template matching (non-parametrik) | MFCC+Δ+ΔΔ per frame |
| **GMM** | Generatif, bag-of-frames (tanpa urutan waktu) | MFCC+Δ+ΔΔ per frame |
| **SVM** | Diskriminatif | Statistik MFCC fixed-length (156 dim) |

Semua model disimpan sebagai bundle `.pkl` di `/content/models/` dan dipakai langsung oleh aplikasi di folder `Deployment/Models/`.

In [ ]:
!pip install hmmlearn

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from hmmlearn import hmm
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Kelas diambil dari data: 4 command (SynTTS) + Unknown (noise_only).
# Kelas yang tidak ada di data otomatis dilewati.
ALL_CLASSES = [
    "Play",
    "Pause",
    "Next_track",
    "Previous_track",
    "Unknown"
]

class_names = [
    label for label in ALL_CLASSES
    if label in set(train_df["label"])
]

label_to_idx = {
    label: idx
    for idx, label in enumerate(class_names)
}

MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

print(label_to_idx)

##Shared module: `classical.py`

Cell berikut adalah salinan **persis** dari `Deployment/classical.py` (feature extraction + inference untuk keempat model). Notebook dan aplikasi deployment memakai kode yang sama, sehingga hasil evaluasi di sini sama dengan perilaku aplikasi. Jika file tersebut diubah, salin ulang ke cell ini.

In [ ]:
# ============================================================
# Classical speech recognizers: feature extraction + inference
#
# File ini juga disalin apa adanya ke notebook training
# (Model_Train/Speech_Recognition_project.ipynb). Jika diubah,
# salin ulang ke notebook dan latih ulang model.
#
# Input semua fungsi: audio 16 kHz yang sudah di-normalize dan
# di-pad/trim menjadi 1 detik (preprocess_audio).
# ============================================================

import glob
import os

import joblib
import librosa
import numpy as np


TARGET_SR = 16000

N_FFT = 512
HOP_LENGTH = 160
WIN_LENGTH = 400

N_MFCC_CLASSICAL = 13
TRIM_TOP_DB = 30
MIN_SPEECH_SAMPLES = int(0.1 * TARGET_SR)   # minimal 100 ms setelah trim
N_SEGMENTS = 3

DEFAULT_CONFIDENCE_THRESHOLD = 0.80

MODEL_ORDER = ["gmm_hmm", "dtw_knn", "gmm", "svm"]


# ============================================================
# Features
# ============================================================

def mfcc_deltas(audio):
    """
    Trim silence, lalu MFCC 13 + delta + delta-delta.

    Output: shape = (39, time_frames), belum dinormalisasi
    """

    trimmed, _ = librosa.effects.trim(
        audio,
        top_db=TRIM_TOP_DB
    )

    if len(trimmed) >= MIN_SPEECH_SAMPLES:
        audio = trimmed

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=TARGET_SR,
        n_mfcc=N_MFCC_CLASSICAL,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH
    )

    delta1 = librosa.feature.delta(mfcc, order=1)
    delta2 = librosa.feature.delta(mfcc, order=2)

    return np.vstack([mfcc, delta1, delta2])


def frame_features(audio):
    """
    Fitur per frame untuk GMM-HMM, GMM, dan DTW.
    CMVN per utterance (normalisasi mean & variance per koefisien).

    Output: shape = (time_frames, 39)
    """

    features = mfcc_deltas(audio)

    features = (
        features - features.mean(axis=1, keepdims=True)
    ) / (
        features.std(axis=1, keepdims=True) + 1e-8
    )

    return features.T.astype(np.float64)


def summary_features(audio):
    """
    Fitur fixed-length untuk SVM (tanpa CMVN, karena CMVN membuat
    mean menjadi 0). Mean dari 3 segmen waktu menjaga urutan bunyi
    (mis. "next" vs "previous"), ditambah std keseluruhan.

    Output: shape = (39 * N_SEGMENTS + 39,) = (156,)
    """

    features = mfcc_deltas(audio)

    segment_means = [
        segment.mean(axis=1)
        for segment in np.array_split(features, N_SEGMENTS, axis=1)
    ]

    return np.concatenate(
        segment_means + [features.std(axis=1)]
    ).astype(np.float64)


FEATURE_FUNCTIONS = {
    "frame": frame_features,
    "summary": summary_features
}


def softmax(x):

    x = np.asarray(x, dtype=np.float64)
    x = x - x.max()

    exp = np.exp(x)

    return exp / exp.sum()


# ============================================================
# Recognizers
#
# Bundle (disimpan dengan joblib):
#   {
#     "type": "gmm_hmm" | "dtw_knn" | "gmm" | "svm",
#     "display_name": str,
#     "class_names": [...],
#     "config": {...},
#     "model": {...}   # objek hmmlearn/sklearn/numpy, isi per type
#   }
# ============================================================

class Recognizer:

    feature_kind = "frame"

    def __init__(self, bundle):

        self.bundle = bundle
        self.type = bundle["type"]
        self.display_name = bundle["display_name"]
        self.class_names = list(bundle["class_names"])
        self.config = bundle.get("config", {})
        self.model = bundle["model"]

        self.confidence_threshold = self.config.get(
            "confidence_threshold",
            DEFAULT_CONFIDENCE_THRESHOLD
        )

    def features(self, audio):
        return FEATURE_FUNCTIONS[self.feature_kind](audio)

    def proba_from_features(self, features):
        raise NotImplementedError

    def predict_proba(self, audio):
        return self.proba_from_features(self.features(audio))

    def proba_batch(self, feature_list):
        return np.array([
            self.proba_from_features(f) for f in feature_list
        ])


class GmmHmmRecognizer(Recognizer):
    """
    model = {"models": {class_name: hmmlearn GMMHMM}}
    Confidence = softmax atas log-likelihood per frame.
    """

    def proba_from_features(self, features):

        scores = [
            self.model["models"][name].score(features)
            for name in self.class_names
        ]

        return softmax(np.array(scores) / len(features))


class GmmRecognizer(Recognizer):
    """
    model = {"models": {class_name: sklearn GaussianMixture}}
    GaussianMixture.score = rata-rata log-likelihood per frame.
    """

    def proba_from_features(self, features):

        scores = [
            self.model["models"][name].score(features)
            for name in self.class_names
        ]

        return softmax(scores)


def dtw_distance(a, b):
    """
    Jarak DTW (euclidean) antara dua sequence (T, 39),
    dinormalisasi dengan panjang warping path.
    """

    D, wp = librosa.sequence.dtw(
        X=a.T,
        Y=b.T,
        metric="euclidean"
    )

    return D[-1, -1] / len(wp)


class DtwKnnRecognizer(Recognizer):
    """
    model = {
        "templates": [array (T, 39), ...],
        "labels": array (n_templates,) index kelas,
        "k": int
    }
    Confidence = vote kNN berbobot 1 / jarak.
    """

    def distances(self, features):
        return np.array([
            dtw_distance(features, template)
            for template in self.model["templates"]
        ])

    def proba_from_distances(self, distances, k=None):

        k = k or self.model["k"]

        nearest = np.argsort(distances)[:k]

        votes = np.zeros(len(self.class_names))

        for i in nearest:
            votes[self.model["labels"][i]] += 1.0 / (distances[i] + 1e-8)

        return votes / votes.sum()

    def proba_from_features(self, features):
        return self.proba_from_distances(self.distances(features))


class SvmRecognizer(Recognizer):
    """
    model = {"pipeline": sklearn Pipeline(StandardScaler, SVC(probability=True))}
    Dilatih dengan label index kelas (0..n_classes-1).
    """

    feature_kind = "summary"

    def proba_from_features(self, features):

        pipeline = self.model["pipeline"]

        proba = pipeline.predict_proba(features[None, :])[0]

        result = np.zeros(len(self.class_names))
        result[pipeline.classes_] = proba

        return result

    def proba_batch(self, feature_list):

        pipeline = self.model["pipeline"]

        proba = pipeline.predict_proba(np.array(feature_list))

        result = np.zeros((len(feature_list), len(self.class_names)))
        result[:, pipeline.classes_] = proba

        return result


RECOGNIZERS = {
    "gmm_hmm": GmmHmmRecognizer,
    "dtw_knn": DtwKnnRecognizer,
    "gmm": GmmRecognizer,
    "svm": SvmRecognizer
}


def build_recognizer(bundle):
    return RECOGNIZERS[bundle["type"]](bundle)


def library_versions():

    import hmmlearn
    import sklearn

    return {
        "numpy": np.__version__,
        "librosa": librosa.__version__,
        "scikit-learn": sklearn.__version__,
        "hmmlearn": hmmlearn.__version__
    }


def load_models(models_dir):
    """
    Load semua bundle *.pkl di models_dir.

    Output: {type: Recognizer}, urut sesuai MODEL_ORDER
    """

    recognizers = {}

    current_versions = library_versions()

    for path in glob.glob(os.path.join(models_dir, "*.pkl")):

        try:
            bundle = joblib.load(path)
            recognizer = build_recognizer(bundle)

        except Exception as e:
            print(f"Failed to load {path}: {e}")
            continue

        trained_versions = recognizer.config.get("versions", {})

        for lib, version in trained_versions.items():
            if current_versions.get(lib) != version:
                print(
                    f"Warning: {os.path.basename(path)} trained with "
                    f"{lib} {version}, running {current_versions.get(lib)}"
                )

        recognizers[recognizer.type] = recognizer

    return {
        key: recognizers[key]
        for key in sorted(
            recognizers,
            key=lambda k: MODEL_ORDER.index(k) if k in MODEL_ORDER else len(MODEL_ORDER)
        )
    }

##Feature extraction

- **Frame features** `(T, 39)`: trim silence → MFCC 13 + Δ + ΔΔ → CMVN per utterance. Dipakai GMM-HMM, DTW, GMM.
- **Summary features** `(156,)`: mean 3 segmen waktu + std keseluruhan (tanpa CMVN). Dipakai SVM.

Untuk Unknown (rekaman noise ~20 detik) diambil potongan 1 detik secara acak dari setiap file.

Fitur dihitung sekali dan di-cache.

In [ ]:
def build_features(dataframe):
    """
    Unknown (noise, ~20 detik): diambil potongan 1 detik secara acak
    (random_crop) agar variasi noise dari seluruh rekaman terpakai.
    Command: 1 detik pertama seperti biasa.
    """

    frames = []
    summaries = []

    for file_path, label in zip(dataframe["file"], dataframe["label"]):

        audio = preprocess_audio(
            file_path,
            label=label,
            random_crop=True
        )

        frames.append(frame_features(audio))
        summaries.append(summary_features(audio))

    labels = dataframe["label"].map(label_to_idx).values

    return frames, np.array(summaries), labels


start = time.time()

# Seed agar random crop Unknown bisa direproduksi
np.random.seed(42)

F_train, S_train, y_train = build_features(train_df)
F_val, S_val, y_val = build_features(val_df)
F_test, S_test, y_test = build_features(test_df)

print(f"Feature extraction selesai ({time.time() - start:.1f}s)")
print("Train:", len(F_train))
print("Val  :", len(F_val))
print("Test :", len(F_test))

In [ ]:
lengths = np.array([len(f) for f in F_train])

print("Frame features   :", F_train[0].shape)
print("Summary features :", S_train.shape)
print("Frames min/mean/max:", lengths.min(), lengths.mean(), lengths.max())

all_frames = F_train + F_val + F_test

print("NaN :", any(np.isnan(f).any() for f in all_frames) or np.isnan(S_train).any())
print("Inf :", any(np.isinf(f).any() for f in all_frames) or np.isinf(S_train).any())

##Helper: bundle, evaluasi, latency

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RESULTS = {}


def make_bundle(model_type, display_name, model, config):

    return {
        "type": model_type,
        "display_name": display_name,
        "class_names": class_names,
        "config": {
            **config,
            "confidence_threshold": DEFAULT_CONFIDENCE_THRESHOLD,
            "versions": library_versions()
        },
        "model": model
    }


def save_bundle(bundle):

    path = os.path.join(MODELS_DIR, bundle["type"] + ".pkl")

    joblib.dump(bundle, path)

    print("Model saved:", path)


def measure_latency(recognizer, n_files=30):
    """
    Rata-rata waktu inference per window 1 detik (termasuk
    feature extraction), seperti di aplikasi real-time.
    Aplikasi memprediksi setiap 250 ms.
    """

    files = test_df["file"].iloc[:n_files]
    audios = [preprocess_audio(f) for f in files]

    start = time.time()

    for audio in audios:
        recognizer.predict_proba(audio)

    return (time.time() - start) / len(audios) * 1000


def evaluate(recognizer, feature_list):

    name = recognizer.display_name

    y_pred = np.argmax(recognizer.proba_batch(feature_list), axis=1)

    result = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "latency_ms": measure_latency(recognizer)
    }

    RESULTS[name] = result

    print(f"========== TEST RESULTS: {name} ==========")
    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1-score : {result['f1']:.4f}")
    print(f"Latency  : {result['latency_ms']:.1f} ms / window")
    print()

    print(
        classification_report(
            y_test,
            y_pred,
            labels=np.arange(len(class_names)),
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(class_names))
    )

    plt.figure(figsize=(7, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - {name}")

    plt.tight_layout()
    plt.show()

    return result


def val_accuracy(recognizer, feature_list):

    y_pred = np.argmax(recognizer.proba_batch(feature_list), axis=1)

    return (y_pred == y_val).mean()

#Model 1: GMM-HMM

Setiap kelas dimodelkan dengan satu **GMM-HMM**. Saat prediksi, audio diberi skor oleh kelima model dan kelas dengan log-likelihood tertinggi dipilih.

- Command (Play, Pause, Next_track, Previous_track): HMM **left-to-right**
- Unknown (jika ada di dataset): HMM **ergodic** (filler model) karena isinya ucapan/noise bebas

In [ ]:
# Regularisasi (prior) agar EM stabil.
# Tanpa ini, state/mixture yang tidak pernah dikunjungi menghasilkan
# pembagian 0/0 di M-step -> parameter NaN.
# - COVARS_FLOOR : batas bawah variance (fitur sudah CMVN, variance ~1)
# - MEANS_WEIGHT : menarik mean ke 0 (prior) bila datanya kosong
# - WEIGHTS_PRIOR: pseudocount Dirichlet untuk bobot mixture
# - TRANS_PRIOR  : pseudocount hanya di transisi yang diizinkan
COVARS_FLOOR = 0.05
MEANS_WEIGHT = 1e-2
WEIGHTS_PRIOR = 2.0
TRANS_PRIOR = 2.0

PRIORS = dict(
    covars_prior=0.0,
    covars_weight=COVARS_FLOOR,
    means_weight=MEANS_WEIGHT,
    weights_prior=WEIGHTS_PRIOR
)


def left_to_right_hmm(sequences, n_states, n_mix, n_iter, random_state):
    """
    HMM left-to-right: state i hanya bisa tetap (self-loop)
    atau pindah ke state i+1. Start selalu di state 0.

    Inisialisasi "flat start": setiap sequence dibagi rata menjadi
    n_states segmen, lalu GMM tiap state di-fit dari frame segmennya.
    Tanpa ini (k-means bawaan hmmlearn), urutan state tidak sesuai
    urutan waktu dan EM bisa membuat sebagian state tidak pernah
    dikunjungi -> parameter NaN.

    init_params="" agar hmmlearn tidak menimpa inisialisasi ini.
    Entri transmat_ bernilai 0 tetap 0 selama EM (prior 1 = tanpa
    pseudocount), sehingga topologi terjaga.
    """

    allowed = np.eye(n_states) + np.eye(n_states, k=1)

    transmat = allowed / allowed.sum(axis=1, keepdims=True)
    transmat_prior = 1.0 + (TRANS_PRIOR - 1.0) * allowed

    model = hmm.GMMHMM(
        n_components=n_states,
        n_mix=n_mix,
        covariance_type="diag",
        **PRIORS,
        transmat_prior=transmat_prior,
        n_iter=n_iter,
        init_params="",
        params="stmcw",
        random_state=random_state
    )

    startprob = np.zeros(n_states)
    startprob[0] = 1.0

    n_features = sequences[0].shape[1]

    means = np.zeros((n_states, n_mix, n_features))
    covars = np.zeros((n_states, n_mix, n_features))
    weights = np.zeros((n_states, n_mix))

    for state in range(n_states):

        frames = np.concatenate([
            np.array_split(x, n_states)[state]
            for x in sequences
        ])

        gmm = GaussianMixture(
            n_components=n_mix,
            covariance_type="diag",
            reg_covar=1e-3,
            random_state=random_state
        ).fit(frames)

        means[state] = gmm.means_
        covars[state] = gmm.covariances_
        weights[state] = gmm.weights_

    model.startprob_ = startprob
    model.transmat_ = transmat
    model.means_ = means
    model.covars_ = covars
    model.weights_ = weights

    return model


def ergodic_hmm(n_states, n_mix, n_iter, random_state):

    return hmm.GMMHMM(
        n_components=n_states,
        n_mix=n_mix,
        covariance_type="diag",
        **PRIORS,
        transmat_prior=TRANS_PRIOR,
        n_iter=n_iter,
        random_state=random_state
    )


def train_gmm_hmm(n_states, n_mix, n_iter=25, random_state=42):

    models = {}

    for idx, name in enumerate(class_names):

        sequences = [f for f, label in zip(F_train, y_train) if label == idx]

        X_concat = np.concatenate(sequences)
        lengths = [len(f) for f in sequences]

        if name == "Unknown":
            model = ergodic_hmm(n_states, n_mix, n_iter, random_state)
        else:
            model = left_to_right_hmm(
                sequences, n_states, n_mix, n_iter, random_state
            )

        model.fit(X_concat, lengths)

        models[name] = model

    return models

In [ ]:
# Hyperparameter search (validation set): jumlah state x Gaussian per state
N_STATES_GRID = [5, 8, 10]
N_MIX_GRID = [2, 4, 8]

best_val_acc = -1.0
best_hmm_bundle = None

for n_states in N_STATES_GRID:
    for n_mix in N_MIX_GRID:

        start = time.time()

        models = train_gmm_hmm(n_states, n_mix)

        bundle = make_bundle(
            "gmm_hmm",
            "GMM-HMM",
            {"models": models},
            {"n_states": n_states, "n_mix": n_mix}
        )

        acc = val_accuracy(build_recognizer(bundle), F_val)

        # Iterasi EM terbanyak; == n_iter berarti belum konvergen (tol)
        max_iter = max(m.monitor_.iter for m in models.values())

        print(
            f"states={n_states:2d} | mix={n_mix} "
            f"| Val Acc: {acc:.4f} "
            f"| EM iters: {max_iter:2d} "
            f"| Time: {time.time() - start:.1f}s"
        )

        if acc > best_val_acc:
            best_val_acc = acc
            best_hmm_bundle = bundle

print("\nBest config:", best_hmm_bundle["config"])
print(f"Best Val Acc: {best_val_acc:.4f}")

In [ ]:
# Cek topologi left-to-right tetap terjaga setelah training
print("Transition matrix - Play:")
print(np.round(best_hmm_bundle["model"]["models"]["Play"].transmat_, 3))

In [ ]:
save_bundle(best_hmm_bundle)

gmm_hmm_result = evaluate(build_recognizer(best_hmm_bundle), F_test)

#Model 2: DTW + kNN

**Dynamic Time Warping** menyelaraskan dua sequence MFCC yang berbeda kecepatan bicara, lalu menghitung jaraknya. Audio baru diklasifikasikan berdasarkan **k template terdekat** (vote berbobot 1/jarak).

Membandingkan dengan semua ~2.800 clip training terlalu lambat untuk real-time (prediksi tiap 250 ms), jadi dipilih beberapa **template representatif per kelas**: clip training dikelompokkan dengan k-means (pada summary features), dan clip yang paling dekat dengan tiap centroid (medoid) dipakai sebagai template.

**Template Unknown lebih banyak dari command.** Noise sangat bervariasi, sehingga butuh banyak template agar noise baru selalu punya tetangga Unknown yang dekat. Sebaliknya, template command yang terlalu banyak justru ikut memilih rekaman command yang tidak biasa, dan template seperti itu mudah "menangkap" noise (noise terbaca sebagai Play/Pause).

**Kriteria pemilihan:** di aplikasi, noise yang terbaca sebagai command (*false trigger*) lebih buruk daripada command yang terlewat (user cukup mengulang). Jadi konfigurasi dipilih berdasarkan **false trigger rate terendah** pada window noise validation, lalu **val accuracy tertinggi**.

In [ ]:
def select_templates(n_command, n_unknown, random_state=42):
    """
    n_command template untuk setiap command, n_unknown untuk Unknown.
    Template = medoid cluster k-means pada summary features.
    """

    templates = []
    labels = []

    for idx, name in enumerate(class_names):

        members = np.where(y_train == idx)[0]
        n = min(n_unknown if name == "Unknown" else n_command, len(members))

        S_members = StandardScaler().fit_transform(S_train[members])

        kmeans = KMeans(
            n_clusters=n,
            n_init=4,
            random_state=random_state
        ).fit(S_members)

        for center in kmeans.cluster_centers_:

            nearest = members[
                np.argmin(np.linalg.norm(S_members - center, axis=1))
            ]

            templates.append(F_train[nearest])
            labels.append(idx)

    return templates, np.array(labels)


def noise_windows(dataframe, crops_per_file=3, seed=7):
    """
    Window noise 1 detik seperti yang dilihat aplikasi real-time:
    beberapa random crop dari setiap file Unknown.
    """

    np.random.seed(seed)

    files = dataframe.loc[dataframe["label"] == "Unknown", "file"]

    return [
        frame_features(
            preprocess_audio(f, label="Unknown", random_crop=True)
        )
        for f in files
        for _ in range(crops_per_file)
    ]


def false_trigger_rate(probas, threshold=DEFAULT_CONFIDENCE_THRESHOLD):
    """
    Persentase window noise yang diprediksi sebagai command dengan
    confidence >= threshold (di aplikasi = command dijalankan).
    """

    if len(probas) == 0:
        return 0.0

    probas = np.asarray(probas)
    is_command = np.array([class_names[i] != "Unknown" for i in probas.argmax(axis=1)])

    return (is_command & (probas.max(axis=1) >= threshold)).mean()


N_val_noise = noise_windows(val_df)
N_test_noise = noise_windows(test_df)

print("Val noise windows :", len(N_val_noise))
print("Test noise windows:", len(N_test_noise))

In [ ]:
# Hyperparameter search (validation set): template command x template Unknown x k
N_COMMAND_TEMPLATES_GRID = [5, 10, 20]
N_UNKNOWN_TEMPLATES_GRID = [60, 120]
K_GRID = [3, 5]

best_key = None
best_dtw_bundle = None

for n_command in N_COMMAND_TEMPLATES_GRID:
    for n_unknown in N_UNKNOWN_TEMPLATES_GRID:

        start = time.time()

        templates, labels = select_templates(n_command, n_unknown)

        recognizer = build_recognizer(make_bundle(
            "dtw_knn",
            "DTW + kNN",
            {"templates": templates, "labels": labels, "k": max(K_GRID)},
            {}
        ))

        # Jarak DTW dihitung sekali, lalu dipakai untuk semua nilai k
        val_distances = [recognizer.distances(f) for f in F_val]
        noise_distances = [recognizer.distances(f) for f in N_val_noise]

        for k in K_GRID:

            y_pred = np.array([
                np.argmax(recognizer.proba_from_distances(d, k))
                for d in val_distances
            ])

            acc = (y_pred == y_val).mean()

            trigger = false_trigger_rate([
                recognizer.proba_from_distances(d, k)
                for d in noise_distances
            ])

            print(
                f"command={n_command:2d} | unknown={n_unknown:3d} | k={k} "
                f"| Val Acc: {acc:.4f} "
                f"| False trigger: {trigger:.4f} "
                f"| Time: {time.time() - start:.1f}s"
            )

            # False trigger terendah dulu, lalu accuracy tertinggi
            key = (trigger, -acc)

            if best_key is None or key < best_key:

                best_key = key
                best_dtw_bundle = make_bundle(
                    "dtw_knn",
                    "DTW + kNN",
                    {"templates": templates, "labels": labels, "k": k},
                    {
                        "n_command_templates": n_command,
                        "n_unknown_templates": n_unknown,
                        "k": k
                    }
                )

print("\nBest config:", best_dtw_bundle["config"])
print(f"Best Val Acc: {-best_key[1]:.4f}")
print(f"Best Val False Trigger: {best_key[0]:.4f}")

In [ ]:
save_bundle(best_dtw_bundle)

dtw_recognizer = build_recognizer(best_dtw_bundle)

dtw_result = evaluate(dtw_recognizer, F_test)

print(
    "False trigger (test noise windows):",
    f"{false_trigger_rate(dtw_recognizer.proba_batch(N_test_noise)):.4f}"
)

#Model 3: GMM (per kelas)

Satu **Gaussian Mixture Model** per kelas, dilatih pada semua frame MFCC kelas tersebut. Berbeda dengan GMM-HMM, urutan waktu diabaikan (*bag of frames*): skor = rata-rata log-likelihood per frame.

In [ ]:
# Hyperparameter search (validation set): jumlah komponen Gaussian
N_COMPONENTS_GRID = [8, 16, 32]

best_val_acc = -1.0
best_gmm_bundle = None

for n_components in N_COMPONENTS_GRID:

    start = time.time()

    models = {}

    for idx, name in enumerate(class_names):

        frames = np.concatenate([
            f for f, label in zip(F_train, y_train) if label == idx
        ])

        models[name] = GaussianMixture(
            n_components=n_components,
            covariance_type="diag",
            reg_covar=1e-3,
            max_iter=200,
            random_state=42
        ).fit(frames)

    bundle = make_bundle(
        "gmm",
        "GMM",
        {"models": models},
        {"n_components": n_components}
    )

    acc = val_accuracy(build_recognizer(bundle), F_val)

    print(
        f"components={n_components:2d} "
        f"| Val Acc: {acc:.4f} "
        f"| Time: {time.time() - start:.1f}s"
    )

    if acc > best_val_acc:
        best_val_acc = acc
        best_gmm_bundle = bundle

print("\nBest config:", best_gmm_bundle["config"])
print(f"Best Val Acc: {best_val_acc:.4f}")

In [ ]:
save_bundle(best_gmm_bundle)

gmm_result = evaluate(build_recognizer(best_gmm_bundle), F_test)

#Model 4: SVM

**Support Vector Machine** (kernel RBF) pada summary features fixed-length (156 dim). Fitur di-standardisasi dengan `StandardScaler` di dalam pipeline. `probability=True` agar confidence bisa dipakai dengan threshold di aplikasi.

In [ ]:
# Hyperparameter search (validation set): C x gamma
C_GRID = [1, 10, 100]
GAMMA_GRID = ["scale", 0.01, 0.001]

best_val_acc = -1.0
best_svm_bundle = None

for C in C_GRID:
    for gamma in GAMMA_GRID:

        start = time.time()

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                kernel="rbf",
                C=C,
                gamma=gamma,
                class_weight="balanced",
                probability=True,
                random_state=42
            ))
        ]).fit(S_train, y_train)

        bundle = make_bundle(
            "svm",
            "SVM",
            {"pipeline": pipeline},
            {"C": C, "gamma": gamma}
        )

        acc = val_accuracy(build_recognizer(bundle), S_val)

        print(
            f"C={C:3d} | gamma={str(gamma):5s} "
            f"| Val Acc: {acc:.4f} "
            f"| Time: {time.time() - start:.1f}s"
        )

        if acc > best_val_acc:
            best_val_acc = acc
            best_svm_bundle = bundle

print("\nBest config:", best_svm_bundle["config"])
print(f"Best Val Acc: {best_val_acc:.4f}")

In [ ]:
save_bundle(best_svm_bundle)

svm_result = evaluate(build_recognizer(best_svm_bundle), S_test)

#Perbandingan Model

`latency_ms` = waktu inference per window 1 detik (termasuk feature extraction). Aplikasi real-time memprediksi setiap **250 ms**, jadi latency harus jauh di bawah itu. DS-CNN (deep learning, versi sebelumnya) ditampilkan sebagai referensi; angka tersebut dari dataset lama (Unknown berbeda), jadi tidak sepenuhnya sebanding.

In [ ]:
comparison = pd.DataFrame(RESULTS).T

comparison.loc["DS-CNN (referensi, dataset lama)"] = {
    "accuracy": 0.9983,
    "precision": 0.9984,
    "recall": 0.9983,
    "f1": 0.9983,
    "latency_ms": np.nan
}

print(comparison.round(4))

In [ ]:
models_only = comparison.drop("DS-CNN (referensi, dataset lama)")

plt.figure(figsize=(8, 4))

plt.bar(models_only.index, models_only["accuracy"])

plt.ylim(0, 1)
plt.ylabel("Test Accuracy")
plt.title("Perbandingan Model Klasik")
plt.xticks(rotation=15)
plt.grid(axis="y")

plt.show()

#Export model ke Deployment

Download `models.zip`, lalu extract isinya ke folder `Deployment/Models/` sehingga berisi:

```
Deployment/Models/gmm_hmm.pkl
Deployment/Models/dtw_knn.pkl
Deployment/Models/gmm.pkl
Deployment/Models/svm.pkl
```

Versi library yang dipakai saat training tersimpan di setiap bundle (`config["versions"]`) dan dicetak di bawah. Pin versi yang sama di `Deployment/requirements.txt`, karena file pickle hmmlearn/scikit-learn bisa gagal di-load pada versi yang berbeda.

In [ ]:
import shutil
from google.colab import files

print(library_versions())

shutil.make_archive("/content/models", "zip", MODELS_DIR)

files.download("/content/models.zip")